# RLHF with PPO: Comparative Evaluation Demo

This notebook demonstrates standard Reinforcement Learning from Human Feedback (RLHF) using:
- **PPO** (Proximal Policy Optimization) for policy learning
- **Comparative evaluation** (Bradley-Terry preference model)
- **Visualizations** of the learned reward landscape

We use a 2D navigation task where the agent must reach a goal while avoiding hazards.
Human feedback is simulated via pairwise trajectory comparisons.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MikeHLee/ai_research/blob/main/topics/high_dimensional_reward_spaces/notebooks/01_rlhf_ppo_comparative_evaluation.ipynb)

In [ ]:
# Environment setup
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    %pip install -q torch matplotlib numpy scipy
    print("Dependencies installed!")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.collections import LineCollection
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Literal
from collections import defaultdict
import random
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Environment: 2D Navigation with Hazards

In [ ]:
class NavigationEnv:
    """2D navigation environment with hazards (black holes) and goal."""
    
    def __init__(
        self,
        goal: np.ndarray = np.array([2.0, 2.0]),
        hazards: List[Tuple[np.ndarray, float]] = None,
        max_steps: int = 150,
        dt: float = 0.1,
    ):
        self.goal = goal
        self.hazards = hazards or [
            (np.array([1.0, 0.5]), 0.3),   # (center, radius)
            (np.array([0.5, 1.5]), 0.25),
            (np.array([1.5, 1.2]), 0.35),
        ]
        self.max_steps = max_steps
        self.dt = dt
        self.obs_dim = 4  # [x, y, vx, vy]
        self.act_dim = 2  # [ax, ay]
        
    def reset(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        self.pos = np.array([0.0, 0.0])
        self.vel = np.array([0.0, 0.0])
        self.steps = 0
        self.trajectory = [self.pos.copy()]
        return self._obs()
    
    def _obs(self):
        return np.concatenate([self.pos, self.vel]).astype(np.float32)
    
    def step(self, action):
        action = np.clip(action, -1.0, 1.0)
        self.vel = np.clip(self.vel + action * self.dt, -2.0, 2.0)
        self.pos = self.pos + self.vel * self.dt
        self.steps += 1
        self.trajectory.append(self.pos.copy())
        
        # Check hazards
        in_hazard = any(
            np.linalg.norm(self.pos - c) < r 
            for c, r in self.hazards
        )
        
        # Rewards (ground truth - not shown to reward model)
        dist = np.linalg.norm(self.pos - self.goal)
        reached_goal = dist < 0.2
        
        info = {
            'in_hazard': in_hazard,
            'reached_goal': reached_goal,
            'distance_to_goal': dist,
        }
        
        done = reached_goal or in_hazard or self.steps >= self.max_steps
        return self._obs(), 0.0, done, info  # Reward=0 (learned from preferences)
    
    def get_trajectory(self):
        return np.array(self.trajectory)

## 2. Trajectory Data Structures

In [ ]:
@dataclass
class Trajectory:
    """A complete trajectory with metadata."""
    id: str
    observations: np.ndarray  # (T, obs_dim)
    actions: np.ndarray       # (T, act_dim)
    path: np.ndarray          # (T, 2) - positions for visualization
    reached_goal: bool
    hit_hazard: bool
    total_steps: int
    
    def __hash__(self):
        return hash(self.id)


@dataclass 
class PreferencePair:
    """A pairwise preference comparison."""
    trajectory_a: Trajectory
    trajectory_b: Trajectory
    preference: Literal['a', 'b', 'equal']  # Which is preferred
    confidence: float = 1.0  # How confident (0-1)
    

@dataclass
class PreferenceDataset:
    """Dataset of preference comparisons."""
    pairs: List[PreferencePair] = field(default_factory=list)
    
    def add(self, pair: PreferencePair):
        self.pairs.append(pair)
    
    def __len__(self):
        return len(self.pairs)

## 3. Simulated Human Preferences

We simulate human preferences based on:
1. **Goal reaching** (most important)
2. **Hazard avoidance** (critical)
3. **Efficiency** (fewer steps is better)

In [ ]:
class SimulatedHuman:
    """Simulates human preferences over trajectories."""
    
    def __init__(
        self,
        goal_weight: float = 10.0,
        hazard_penalty: float = -50.0,
        efficiency_weight: float = 0.1,
        noise_level: float = 0.1,  # Probability of random preference
    ):
        self.goal_weight = goal_weight
        self.hazard_penalty = hazard_penalty
        self.efficiency_weight = efficiency_weight
        self.noise_level = noise_level
    
    def _score(self, traj: Trajectory) -> float:
        """Internal scoring function (hidden from reward model)."""
        score = 0.0
        
        # Goal bonus
        if traj.reached_goal:
            score += self.goal_weight
        
        # Hazard penalty
        if traj.hit_hazard:
            score += self.hazard_penalty
        
        # Efficiency (negative steps)
        score -= self.efficiency_weight * traj.total_steps
        
        return score
    
    def compare(self, traj_a: Trajectory, traj_b: Trajectory) -> PreferencePair:
        """Compare two trajectories and return preference."""
        score_a = self._score(traj_a)
        score_b = self._score(traj_b)
        
        # Add noise
        if random.random() < self.noise_level:
            preference = random.choice(['a', 'b', 'equal'])
        else:
            diff = score_a - score_b
            if abs(diff) < 0.5:  # Close scores
                preference = 'equal'
            elif diff > 0:
                preference = 'a'
            else:
                preference = 'b'
        
        confidence = min(1.0, abs(score_a - score_b) / 10.0)
        return PreferencePair(traj_a, traj_b, preference, confidence)

## 4. Reward Model (Bradley-Terry)

The reward model learns from pairwise comparisons using the Bradley-Terry model:

$$P(\tau_a \succ \tau_b) = \sigma(R(\tau_a) - R(\tau_b))$$

where $\sigma$ is the sigmoid function.

In [ ]:
class RewardModel(nn.Module):
    """Neural network that predicts reward from trajectory."""
    
    def __init__(self, obs_dim: int, hidden_dim: int = 64):
        super().__init__()
        # Trajectory encoder (processes sequence of observations)
        self.encoder = nn.LSTM(obs_dim, hidden_dim, batch_first=True)
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, trajectory: torch.Tensor) -> torch.Tensor:
        """Predict reward for a trajectory.
        
        Args:
            trajectory: (batch, seq_len, obs_dim) or (seq_len, obs_dim)
        Returns:
            reward: (batch, 1) or (1,)
        """
        if trajectory.dim() == 2:
            trajectory = trajectory.unsqueeze(0)
        
        # Encode trajectory
        _, (h_n, _) = self.encoder(trajectory)
        h = h_n[-1]  # (batch, hidden)
        
        return self.reward_head(h)
    
    def predict_preference(self, traj_a: torch.Tensor, traj_b: torch.Tensor) -> torch.Tensor:
        """Predict P(traj_a > traj_b) using Bradley-Terry model."""
        r_a = self.forward(traj_a)
        r_b = self.forward(traj_b)
        return torch.sigmoid(r_a - r_b)

In [ ]:
def train_reward_model(
    model: RewardModel,
    dataset: PreferenceDataset,
    epochs: int = 100,
    lr: float = 1e-3,
    device: torch.device = device,
) -> List[float]:
    """Train reward model on preference dataset."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        random.shuffle(dataset.pairs)
        
        for pair in dataset.pairs:
            # Convert to tensors
            traj_a = torch.tensor(pair.trajectory_a.observations, dtype=torch.float32, device=device)
            traj_b = torch.tensor(pair.trajectory_b.observations, dtype=torch.float32, device=device)
            
            # Get preference probability
            p_a_better = model.predict_preference(traj_a, traj_b)
            
            # Target
            if pair.preference == 'a':
                target = torch.tensor([[1.0]], device=device)
            elif pair.preference == 'b':
                target = torch.tensor([[0.0]], device=device)
            else:  # equal
                target = torch.tensor([[0.5]], device=device)
            
            # Binary cross-entropy loss
            loss = F.binary_cross_entropy(p_a_better, target)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        losses.append(epoch_loss / len(dataset))
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}: Loss = {losses[-1]:.4f}")
    
    return losses

## 5. Policy Networks (Actor-Critic)

In [ ]:
class Actor(nn.Module):
    """Policy network that outputs action distribution."""
    
    def __init__(self, obs_dim: int, act_dim: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
        )
        self.mu = nn.Linear(hidden, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim))
    
    def forward(self, obs):
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)
        h = self.net(obs)
        mu = torch.tanh(self.mu(h))  # Bound actions
        std = torch.exp(self.log_std.clamp(-20, 2))
        return Normal(mu, std)
    
    def get_action(self, obs, deterministic=False):
        dist = self.forward(obs)
        if deterministic:
            action = dist.mean
        else:
            action = dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return action.squeeze(0), log_prob.squeeze(0)


class Critic(nn.Module):
    """Value network."""
    
    def __init__(self, obs_dim: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )
    
    def forward(self, obs):
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)
        return self.net(obs).squeeze(-1)

## 6. PPO Training with Learned Reward

In [ ]:
def collect_trajectory(env: NavigationEnv, actor: Actor, device: torch.device) -> Trajectory:
    """Collect a single trajectory using the policy."""
    obs = env.reset()
    observations, actions = [obs], []
    done = False
    
    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
        with torch.no_grad():
            action, _ = actor.get_action(obs_t)
        action = action.cpu().numpy()
        
        obs, _, done, info = env.step(action)
        observations.append(obs)
        actions.append(action)
    
    return Trajectory(
        id=f"traj_{random.randint(0, 1000000)}",
        observations=np.array(observations[:-1]),  # Exclude terminal
        actions=np.array(actions),
        path=env.get_trajectory(),
        reached_goal=info['reached_goal'],
        hit_hazard=info['in_hazard'],
        total_steps=len(actions),
    )


def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    """Compute Generalized Advantage Estimation."""
    advantages = []
    gae = 0
    values = list(values) + [0]  # Bootstrap with 0
    
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t + 1] - values[t]
        gae = delta + gamma * lam * gae
        advantages.insert(0, gae)
    
    returns = [adv + val for adv, val in zip(advantages, values[:-1])]
    return advantages, returns

In [ ]:
def train_ppo_episode(
    env: NavigationEnv,
    actor: Actor,
    critic: Critic,
    reward_model: RewardModel,
    actor_optim: torch.optim.Optimizer,
    critic_optim: torch.optim.Optimizer,
    device: torch.device,
    clip_ratio: float = 0.2,
    train_iters: int = 5,
) -> Dict:
    """Run one episode of PPO training."""
    obs = env.reset()
    observations, actions, log_probs, values = [], [], [], []
    done = False
    
    # Collect episode
    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
        with torch.no_grad():
            action, log_prob = actor.get_action(obs_t)
            value = critic(obs_t)
        
        observations.append(obs_t)
        actions.append(action)
        log_probs.append(log_prob)
        values.append(value.item())
        
        obs, _, done, info = env.step(action.cpu().numpy())
    
    # Get rewards from learned reward model
    obs_sequence = torch.stack(observations)  # (T, obs_dim)
    with torch.no_grad():
        # Use reward model to get per-step rewards (approximate)
        trajectory_reward = reward_model(obs_sequence).item()
    
    # Distribute trajectory reward across steps (simple approach)
    T = len(observations)
    rewards = [trajectory_reward / T] * T
    
    # Bonus/penalty shaping
    if info['reached_goal']:
        rewards[-1] += 5.0
    if info['in_hazard']:
        rewards[-1] -= 10.0
    
    # Compute advantages
    advantages, returns = compute_gae(rewards, values)
    advantages = torch.tensor(advantages, dtype=torch.float32, device=device)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    returns = torch.tensor(returns, dtype=torch.float32, device=device)
    
    observations = torch.stack(observations)
    actions = torch.stack(actions)
    old_log_probs = torch.stack(log_probs)
    
    # PPO updates
    for _ in range(train_iters):
        dist = actor(observations)
        new_log_probs = dist.log_prob(actions).sum(-1)
        ratio = torch.exp(new_log_probs - old_log_probs)
        
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()
        
        actor_optim.zero_grad()
        actor_loss.backward()
        actor_optim.step()
        
        critic_loss = F.mse_loss(critic(observations), returns)
        critic_optim.zero_grad()
        critic_loss.backward()
        critic_optim.step()
    
    return {
        'return': sum(rewards),
        'steps': T,
        'reached_goal': info['reached_goal'],
        'hit_hazard': info['in_hazard'],
        'trajectory': env.get_trajectory(),
    }

## 7. Full Training Pipeline

In [ ]:
def run_rlhf_pipeline(
    n_preference_trajectories: int = 100,
    n_comparisons: int = 200,
    n_rl_episodes: int = 200,
    seed: int = 42,
):
    """Full RLHF pipeline: collect preferences, train reward model, train policy."""
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    # Initialize
    env = NavigationEnv()
    human = SimulatedHuman()
    
    # Random policy for initial data collection
    random_actor = Actor(env.obs_dim, env.act_dim).to(device)
    
    print("=" * 50)
    print("Phase 1: Collecting trajectories with random policy")
    print("=" * 50)
    
    trajectories = []
    for i in range(n_preference_trajectories):
        traj = collect_trajectory(env, random_actor, device)
        trajectories.append(traj)
        if (i + 1) % 20 == 0:
            goals = sum(1 for t in trajectories if t.reached_goal)
            hazards = sum(1 for t in trajectories if t.hit_hazard)
            print(f"  Collected {i+1} trajectories: {goals} goals, {hazards} hazards")
    
    print("\n" + "=" * 50)
    print("Phase 2: Collecting human preferences")
    print("=" * 50)
    
    dataset = PreferenceDataset()
    for i in range(n_comparisons):
        # Sample two different trajectories
        traj_a, traj_b = random.sample(trajectories, 2)
        pair = human.compare(traj_a, traj_b)
        dataset.add(pair)
    
    # Analyze preferences
    a_wins = sum(1 for p in dataset.pairs if p.preference == 'a')
    b_wins = sum(1 for p in dataset.pairs if p.preference == 'b')
    ties = sum(1 for p in dataset.pairs if p.preference == 'equal')
    print(f"  Collected {len(dataset)} comparisons: {a_wins} A-wins, {b_wins} B-wins, {ties} ties")
    
    print("\n" + "=" * 50)
    print("Phase 3: Training reward model")
    print("=" * 50)
    
    reward_model = RewardModel(env.obs_dim).to(device)
    rm_losses = train_reward_model(reward_model, dataset, epochs=100)
    
    print("\n" + "=" * 50)
    print("Phase 4: Training policy with PPO")
    print("=" * 50)
    
    actor = Actor(env.obs_dim, env.act_dim).to(device)
    critic = Critic(env.obs_dim).to(device)
    actor_optim = torch.optim.Adam(actor.parameters(), lr=3e-4)
    critic_optim = torch.optim.Adam(critic.parameters(), lr=1e-3)
    
    history = defaultdict(list)
    
    for ep in range(n_rl_episodes):
        result = train_ppo_episode(
            env, actor, critic, reward_model,
            actor_optim, critic_optim, device
        )
        
        for k, v in result.items():
            if k != 'trajectory':
                history[k].append(v)
        
        if (ep + 1) % 20 == 0:
            recent = 20
            avg_return = np.mean(history['return'][-recent:])
            goal_rate = np.mean(history['reached_goal'][-recent:])
            hazard_rate = np.mean(history['hit_hazard'][-recent:])
            print(f"  Episode {ep+1}: Return={avg_return:.1f}, Goals={goal_rate:.0%}, Hazards={hazard_rate:.0%}")
    
    return {
        'env': env,
        'actor': actor,
        'critic': critic,
        'reward_model': reward_model,
        'dataset': dataset,
        'trajectories': trajectories,
        'history': dict(history),
        'rm_losses': rm_losses,
    }

In [ ]:
# Run the pipeline
results = run_rlhf_pipeline(
    n_preference_trajectories=100,
    n_comparisons=200,
    n_rl_episodes=200,
)

## 8. Visualizations

In [ ]:
def visualize_reward_landscape(reward_model, env, device, resolution=50):
    """Visualize the learned reward function over state space."""
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Create grid
    x = np.linspace(-0.5, 2.5, resolution)
    y = np.linspace(-0.5, 2.5, resolution)
    X, Y = np.meshgrid(x, y)
    
    # Compute reward at each point
    Z = np.zeros_like(X)
    for i in range(resolution):
        for j in range(resolution):
            # Create a simple "trajectory" of staying at this point
            obs = np.array([[X[i, j], Y[i, j], 0, 0]] * 5, dtype=np.float32)
            obs_t = torch.tensor(obs, device=device)
            with torch.no_grad():
                Z[i, j] = reward_model(obs_t).item()
    
    # Plot 1: Reward heatmap
    ax1 = axes[0]
    im = ax1.contourf(X, Y, Z, levels=30, cmap='RdYlGn')
    plt.colorbar(im, ax=ax1, label='Learned Reward')
    
    # Add hazards
    for center, radius in env.hazards:
        circle = Circle(center, radius, fill=True, color='black', alpha=0.7)
        ax1.add_patch(circle)
    
    ax1.plot(*env.goal, 'g*', markersize=20, label='Goal')
    ax1.plot(0, 0, 'wo', markersize=10, markeredgecolor='black', label='Start')
    ax1.set_title('Learned Reward Landscape')
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.legend(loc='upper left')
    ax1.set_aspect('equal')
    
    # Plot 2: Training curves
    ax2 = axes[1]
    history = results['history']
    window = 10
    returns = np.convolve(history['return'], np.ones(window)/window, mode='valid')
    ax2.plot(returns, 'b-', linewidth=2)
    ax2.set_title('Training Progress')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Return')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Goal/Hazard rates
    ax3 = axes[2]
    goals = np.convolve(history['reached_goal'], np.ones(window)/window, mode='valid')
    hazards = np.convolve(history['hit_hazard'], np.ones(window)/window, mode='valid')
    ax3.plot(goals, 'g-', linewidth=2, label='Goal Rate')
    ax3.plot(hazards, 'r-', linewidth=2, label='Hazard Rate')
    ax3.set_title('Success/Failure Rates')
    ax3.set_xlabel('Episode')
    ax3.set_ylabel('Rate')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(-0.05, 1.05)
    
    plt.tight_layout()
    plt.savefig('rlhf_ppo_results.png', dpi=150)
    plt.show()

visualize_reward_landscape(results['reward_model'], results['env'], device)

In [ ]:
def visualize_trajectories(env, actor, device, n_trajectories=10):
    """Visualize sample trajectories from the trained policy."""
    
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Draw hazards
    for center, radius in env.hazards:
        circle = Circle(center, radius, fill=True, color='red', alpha=0.3, label='Hazard')
        ax.add_patch(circle)
        circle_edge = Circle(center, radius, fill=False, color='darkred', linewidth=2)
        ax.add_patch(circle_edge)
    
    # Collect and draw trajectories
    colors = plt.cm.viridis(np.linspace(0, 1, n_trajectories))
    
    for i in range(n_trajectories):
        traj = collect_trajectory(env, actor, device)
        path = traj.path
        
        # Color by outcome
        if traj.reached_goal:
            color = 'green'
            alpha = 0.8
        elif traj.hit_hazard:
            color = 'red'
            alpha = 0.8
        else:
            color = 'blue'
            alpha = 0.5
        
        ax.plot(path[:, 0], path[:, 1], color=color, alpha=alpha, linewidth=2)
        ax.plot(path[-1, 0], path[-1, 1], 'o', color=color, markersize=8)
    
    # Goal and start
    ax.plot(*env.goal, 'g*', markersize=25, label='Goal', zorder=10)
    ax.plot(0, 0, 'ko', markersize=15, label='Start', zorder=10)
    
    ax.set_xlim(-0.5, 2.8)
    ax.set_ylim(-0.5, 2.8)
    ax.set_aspect('equal')
    ax.set_title('Trained Policy Trajectories\n(Green=Goal, Red=Hazard, Blue=Timeout)')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.grid(True, alpha=0.3)
    
    # Custom legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='green', linewidth=2, label='Reached Goal'),
        Line2D([0], [0], color='red', linewidth=2, label='Hit Hazard'),
        Line2D([0], [0], color='blue', linewidth=2, label='Timeout'),
    ]
    ax.legend(handles=legend_elements, loc='upper left')
    
    plt.tight_layout()
    plt.savefig('rlhf_trajectories.png', dpi=150)
    plt.show()

visualize_trajectories(results['env'], results['actor'], device, n_trajectories=15)

In [ ]:
def visualize_preference_analysis(dataset, reward_model, device):
    """Analyze how well the reward model captures preferences."""
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Compute predicted vs actual preferences
    correct = 0
    predictions = []
    actuals = []
    
    for pair in dataset.pairs:
        traj_a = torch.tensor(pair.trajectory_a.observations, dtype=torch.float32, device=device)
        traj_b = torch.tensor(pair.trajectory_b.observations, dtype=torch.float32, device=device)
        
        with torch.no_grad():
            p_a = reward_model.predict_preference(traj_a, traj_b).item()
        
        predictions.append(p_a)
        
        if pair.preference == 'a':
            actuals.append(1.0)
            if p_a > 0.5:
                correct += 1
        elif pair.preference == 'b':
            actuals.append(0.0)
            if p_a < 0.5:
                correct += 1
        else:
            actuals.append(0.5)
            if 0.4 < p_a < 0.6:
                correct += 1
    
    accuracy = correct / len(dataset)
    
    # Plot 1: Predicted vs Actual
    ax1 = axes[0]
    ax1.scatter(actuals, predictions, alpha=0.5, c='blue')
    ax1.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect')
    ax1.set_xlabel('Actual Preference (1=A, 0=B, 0.5=Equal)')
    ax1.set_ylabel('Predicted P(A > B)')
    ax1.set_title(f'Preference Prediction\nAccuracy: {accuracy:.1%}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Reward model loss
    ax2 = axes[1]
    ax2.plot(results['rm_losses'], 'b-', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('BCE Loss')
    ax2.set_title('Reward Model Training')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('preference_analysis.png', dpi=150)
    plt.show()
    
    print(f"\nPreference Prediction Accuracy: {accuracy:.1%}")

visualize_preference_analysis(results['dataset'], results['reward_model'], device)

## 9. Summary

This notebook demonstrated standard RLHF with:

1. **Trajectory collection** with random/trained policies
2. **Preference elicitation** via pairwise comparisons (Bradley-Terry model)
3. **Reward model training** from preferences
4. **PPO policy optimization** using learned rewards

### Limitations of This Approach

As discussed in Appendix D of the STRS paper, this approach has blind spots:

| Anomaly | Detectable? | Why? |
|---------|-------------|------|
| **Black Holes** | ⚠️ Partial | Only if we sample trajectories that hit them |
| **Cliffs** | ❌ No | Comparisons don't span discontinuities |
| **Wormholes** | ❌ No | Local comparisons can't detect shortcuts |
| **Plateaus** | ❌ No | No process supervision |

**Next notebook**: SGPO with Process Supervision addresses these limitations.